# Session 1: Modern LLM Foundations for Agentic Systems (Instructor Notebook -- Full Solutions)

This is the **instructor version** of Session 1. It contains the same demos as the student notebook, plus **complete, working solutions** for all four tasks. Use this as your reference during class delivery.

## Learning Objectives

By the end of this session, students will be able to:

1. **Set up and configure** LLM API connections using the OpenAI Python SDK
2. **Understand tokenization** and how context windows affect LLM behavior
3. **Control model output** by tuning parameters such as temperature, top_p, and max_tokens
4. **Use system messages** to shape LLM response behavior and persona
5. **Build basic LLM pipelines** by chaining multiple API calls together
6. **Lay the groundwork** for conversational agents with message history management

In [ ]:
# ============================================================
# Imports and Setup
# ============================================================

import openai
import tiktoken
import os
import json

# Ensure your OpenAI API key is set as an environment variable:
#   export OPENAI_API_KEY="sk-..."
# Or uncomment the line below and paste your key (not recommended for production):
# os.environ["OPENAI_API_KEY"] = "sk-..."

print("All imports successful!")
print(f"OpenAI SDK version: {openai.__version__}")

---
## Demos (Follow Along)

The following five demos are fully coded. Run each cell, observe the output, and make sure you understand what is happening before moving on.

### Demo 1: Setting Up LLM API Connections

In this demo we initialize the OpenAI client and make our first chat completion call. The `client` object handles authentication, request formatting, and response parsing for us.

In [ ]:
# Demo 1 - Setting Up LLM API Connections

# Step 1: Initialize the OpenAI client
# The client automatically reads OPENAI_API_KEY from the environment.
client = openai.OpenAI()

# Step 2: Make a simple chat completion call
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "What is an LLM? Explain in two sentences."}
    ],
    max_tokens=100
)

# Step 3: Print the response
print("Model used :", response.model)
print("Finish reason:", response.choices[0].finish_reason)
print("\nAssistant response:")
print(response.choices[0].message.content)

### Demo 2: Understanding Tokenization and Context Windows

Tokens are the fundamental units that LLMs process. A token can be a word, part of a word, or even a single character. Understanding tokenization helps you:
- Estimate API costs (billing is per-token)
- Stay within context window limits
- Write more efficient prompts

In [ ]:
# Demo 2 - Understanding Tokenization and Context Windows

# Step 1: Get the tokenizer for gpt-4o-mini
encoder = tiktoken.encoding_for_model("gpt-4o-mini")

# Step 2: Encode various texts and count tokens
texts = [
    "Hello, world!",
    "Artificial Intelligence is transforming the way we build software.",
    "The quick brown fox jumps over the lazy dog.",
    "Supercalifragilisticexpialidocious",
    "def fibonacci(n):\n    if n <= 1:\n        return n\n    return fibonacci(n-1) + fibonacci(n-2)"
]

print("Token counts for different texts:")
print("=" * 60)
for text in texts:
    tokens = encoder.encode(text)
    print(f"\nText   : {text[:60]}{'...' if len(text) > 60 else ''}")
    print(f"Chars  : {len(text)}")
    print(f"Tokens : {len(tokens)}")
    print(f"Token IDs (first 10): {tokens[:10]}")

# Step 3: Demonstrate context window limits
print("\n" + "=" * 60)
print("Context window reference:")
print("  gpt-4o-mini : 128,000 tokens")
print("  gpt-4o      : 128,000 tokens")
print("  gpt-3.5-turbo: 16,385 tokens")

# Show how a long text would consume the window
long_text = "This is a sample sentence. " * 500
long_tokens = encoder.encode(long_text)
print(f"\nLong text ({len(long_text)} chars) = {len(long_tokens)} tokens")
print(f"That is {len(long_tokens)/128000*100:.2f}% of the gpt-4o-mini context window.")

### Demo 3: Exploring Model Parameters

LLMs expose several parameters that control the randomness and length of generated text:

| Parameter | Description |
|-----------|-------------|
| `temperature` | Controls randomness. 0 = deterministic, 1 = creative |
| `top_p` | Nucleus sampling. Lower values = more focused output |
| `max_tokens` | Maximum number of tokens in the response |

In [ ]:
# Demo 3 - Exploring Model Parameters

client = openai.OpenAI()
prompt = "Write a one-sentence description of the future of AI."

# --- Temperature comparison ---
print("TEMPERATURE COMPARISON")
print("=" * 60)

for temp in [0.0, 0.5, 1.0]:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=temp,
        max_tokens=80
    )
    print(f"\nTemperature {temp}:")
    print(f"  {response.choices[0].message.content}")

# --- top_p comparison ---
print("\n" + "=" * 60)
print("TOP_P COMPARISON")
print("=" * 60)

for top_p in [0.1, 0.5, 1.0]:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        top_p=top_p,
        max_tokens=80
    )
    print(f"\ntop_p {top_p}:")
    print(f"  {response.choices[0].message.content}")

# --- max_tokens comparison ---
print("\n" + "=" * 60)
print("MAX_TOKENS COMPARISON")
print("=" * 60)

for max_tok in [10, 30, 100]:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.5,
        max_tokens=max_tok
    )
    text = response.choices[0].message.content
    print(f"\nmax_tokens={max_tok} (finish_reason={response.choices[0].finish_reason}):")
    print(f"  {text}")

### Demo 4: Comparing LLM Response Behaviors

The **system message** is a powerful lever for controlling how the LLM behaves. By changing the system message, you can make the same model act as different "personas" or agents, each with its own tone, expertise, and communication style.

In [ ]:
# Demo 4 - Comparing LLM Response Behaviors

client = openai.OpenAI()

user_question = "Explain what a REST API is."

personas = {
    "Formal Academic": (
        "You are a formal academic professor. Use precise technical language, "
        "cite concepts from computer science, and maintain a scholarly tone."
    ),
    "Casual Friendly": (
        "You are a casual, friendly mentor. Use simple language, analogies, "
        "and a conversational tone. Imagine you are explaining to a friend."
    ),
    "Technical DevOps Engineer": (
        "You are a senior DevOps engineer. Focus on practical implementation details, "
        "mention specific tools and protocols, and be concise."
    ),
}

print(f"User Question: {user_question}")
print("=" * 60)

for persona_name, system_msg in personas.items():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_question}
        ],
        temperature=0.7,
        max_tokens=200
    )
    print(f"\n--- {persona_name} ---")
    print(response.choices[0].message.content)
    print()

### Demo 5: Building a Basic LLM Pipeline

A pipeline chains multiple LLM calls together, where the output of one step becomes the input of the next. This pattern is fundamental to agentic systems, where an LLM may need to plan, execute, and reflect across multiple steps.

In [ ]:
# Demo 5 - Building a Basic LLM Pipeline

client = openai.OpenAI()

def call_llm(system_message, user_message, temperature=0.5, max_tokens=300):
    """Helper function to make an LLM call with a system and user message."""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message}
        ],
        temperature=temperature,
        max_tokens=max_tokens
    )
    return response.choices[0].message.content


# --- Pipeline: Summarize -> Extract Key Points ---

original_text = """
Large Language Models (LLMs) have rapidly evolved from simple text generators to sophisticated 
reasoning engines. Modern LLMs like GPT-4, Claude, and Gemini can understand context, follow 
complex instructions, and even use tools. This evolution has enabled the emergence of agentic AI 
systems -- autonomous programs that can plan, reason, and take actions to accomplish goals. These 
agents leverage LLMs as their core reasoning component while integrating with external tools, 
databases, and APIs. The key challenges in building agentic systems include managing context 
windows, ensuring reliable tool use, implementing effective guardrails, and maintaining 
conversational state across multi-turn interactions. As the field progresses, we are seeing 
the rise of multi-agent architectures where multiple specialized agents collaborate to solve 
complex problems that no single agent could handle alone.
"""

print("PIPELINE STEP 1: Summarize")
print("=" * 60)

summary = call_llm(
    system_message="You are a concise summarizer. Summarize the given text in 2-3 sentences.",
    user_message=f"Summarize this text:\n\n{original_text}"
)
print(summary)

print("\nPIPELINE STEP 2: Extract Key Points")
print("=" * 60)

key_points = call_llm(
    system_message="You are a structured information extractor. Extract exactly 5 key points as a numbered list.",
    user_message=f"Extract the key points from this summary:\n\n{summary}"
)
print(key_points)

print("\nPIPELINE RESULT: Complete output")
print("=" * 60)
print(f"Original length : {len(original_text.split())} words")
print(f"Summary length  : {len(summary.split())} words")
print(f"\nKey Points:\n{key_points}")

---
## Tasks -- Full Solutions

Below are the complete, working solutions for all four student tasks. Each solution is preceded by an explanation of the approach.

### Task 1: Configure and Test Multiple LLM Provider Connections

**Approach:** The solution creates an OpenAI client, sends a minimal test message with constrained `max_tokens`, and returns the response content. We also wrap the call in a try/except so students can see how to handle connection errors gracefully. The function prints a success/failure message and returns the response text.

In [ ]:
# Task 1 - SOLUTION: Configure and Test Multiple LLM Provider Connections

def test_llm_connection():
    """Initialize OpenAI client and test with a simple message."""
    # Step 1: Create the client. It reads OPENAI_API_KEY from the environment.
    client = openai.OpenAI()

    try:
        # Step 2: Send a minimal test message.
        # We use max_tokens=50 to keep the response short and fast.
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "user", "content": "Hello! Respond with a single sentence."}
            ],
            max_tokens=50
        )

        # Step 3: Extract and return the response text.
        result = response.choices[0].message.content
        print(f"Connection successful! Response: {result}")
        return result

    except openai.AuthenticationError:
        print("ERROR: Invalid API key. Check your OPENAI_API_KEY environment variable.")
        return None
    except openai.APIConnectionError:
        print("ERROR: Could not connect to the OpenAI API. Check your network.")
        return None
    except Exception as e:
        print(f"ERROR: Unexpected error -- {e}")
        return None


# Test the function
result = test_llm_connection()
print(f"\nReturned value: {result}")

### Task 2: Analyze Token Usage and Optimize Prompt Length

**Approach:** We use `tiktoken` to get the encoder for the target model, then count the tokens in the input prompt. If the count exceeds the limit, we split the text into sentences (using `". "` as a delimiter), and iteratively add sentences back until we would exceed the budget. This preserves readability by never cutting mid-sentence. The function returns a dictionary with the original count, the optimized text, and the new token count.

In [ ]:
# Task 2 - SOLUTION: Analyze Token Usage and Optimize Prompt Length

def optimize_prompt(prompt, max_tokens=500):
    """Count tokens and truncate at sentence boundaries if over the limit.

    Returns a dict with:
        - original_tokens: token count of the input prompt
        - optimized_text: the (possibly truncated) prompt
        - optimized_tokens: token count of the optimized prompt
        - was_truncated: whether truncation was applied
    """
    # Step 1: Get the encoder for the target model.
    encoder = tiktoken.encoding_for_model("gpt-4o-mini")

    # Step 2: Count tokens in the original prompt.
    original_tokens = len(encoder.encode(prompt))

    # Step 3: If under the limit, return as-is.
    if original_tokens <= max_tokens:
        return {
            "original_tokens": original_tokens,
            "optimized_text": prompt,
            "optimized_tokens": original_tokens,
            "was_truncated": False
        }

    # Step 4: Split into sentences and rebuild until we hit the budget.
    sentences = prompt.split(". ")
    optimized_parts = []
    running_token_count = 0

    for sentence in sentences:
        # Add the period back (it was removed by split) except for the last fragment.
        candidate = sentence if sentence.endswith(".") else sentence + "."
        candidate_tokens = len(encoder.encode(candidate))

        if running_token_count + candidate_tokens > max_tokens:
            break  # Adding this sentence would exceed the limit.

        optimized_parts.append(candidate)
        running_token_count += candidate_tokens

    optimized_text = " ".join(optimized_parts)
    optimized_tokens = len(encoder.encode(optimized_text))

    return {
        "original_tokens": original_tokens,
        "optimized_text": optimized_text,
        "optimized_tokens": optimized_tokens,
        "was_truncated": True
    }


# Test the function
long_prompt = (
    "Artificial intelligence is changing the world. "
    "Machine learning models can now process natural language. "
    "Deep learning has enabled breakthroughs in computer vision. "
    "Transformers are the architecture behind modern LLMs. "
    "Attention mechanisms allow models to focus on relevant parts of the input. "
) * 20  # Repeat to make it long

result = optimize_prompt(long_prompt, max_tokens=50)
print(f"Original tokens : {result['original_tokens']}")
print(f"Optimized tokens: {result['optimized_tokens']}")
print(f"Was truncated   : {result['was_truncated']}")
print(f"\nOptimized text:\n{result['optimized_text']}")

### Task 3: Experiment with Model Parameters

**Approach:** We loop over three temperature values (0.0, 0.5, 1.0), sending the exact same prompt and model each time. Results are stored in a dictionary keyed by temperature. We also print the results in a formatted comparison so the differences are easy to spot. Using `temperature=0.0` should give nearly identical outputs on repeated runs, while `temperature=1.0` will produce more varied and creative responses.

In [ ]:
# Task 3 - SOLUTION: Experiment with Model Parameters

def compare_temperatures(prompt):
    """Send the same prompt at different temperatures and return all responses.

    Returns a dictionary: {temperature_value: response_text}
    """
    # Step 1: Initialize the client.
    client = openai.OpenAI()

    # Step 2: Define the temperatures to compare.
    temperatures = [0.0, 0.5, 1.0]
    results = {}

    # Step 3: Loop over temperatures, making one API call per setting.
    for temp in temperatures:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=temp,
            max_tokens=150
        )
        results[temp] = response.choices[0].message.content

    return results


# Test the function
test_prompt = "Describe the color blue in a creative way."
results = compare_temperatures(test_prompt)

print(f"Prompt: {test_prompt}")
print("=" * 60)
for temp, text in results.items():
    print(f"\nTemperature {temp}:")
    print(f"  {text}")
    print()

### Task 4: Build a Simple Conversational Agent Foundation

**Approach:** The `SimpleAgent` class maintains an internal `messages` list that starts with a system message defining the agent's persona. Each call to `chat()` appends the user message to the history, sends the full history to the LLM (so the model has context from previous turns), and then appends the assistant's response to the history before returning it. The `get_history()` method provides access to the full conversation log. This pattern is the foundation of every multi-turn conversational agent.

In [ ]:
# Task 4 - SOLUTION: Build a Simple Conversational Agent Foundation

class SimpleAgent:
    def __init__(self):
        """Initialize the agent with a system message and empty history."""
        # Step 1: Create the OpenAI client.
        self.client = openai.OpenAI()

        # Step 2: Define the system message that sets the agent's persona.
        # This message persists across the entire conversation.
        self.system_message = (
            "You are a helpful coding assistant. You provide clear, concise explanations "
            "of programming concepts and write clean, well-commented code examples. "
            "When asked a question, you first explain the concept, then show an example."
        )

        # Step 3: Initialize the message history with the system message.
        self.messages = [
            {"role": "system", "content": self.system_message}
        ]

    def chat(self, user_input):
        """Process a user message and return the assistant's response."""
        # Step 1: Append the user's message to the conversation history.
        self.messages.append({"role": "user", "content": user_input})

        # Step 2: Send the FULL history to the LLM.
        # This is what gives the model context of previous turns.
        response = self.client.chat.completions.create(
            model="gpt-4o-mini",
            messages=self.messages,
            temperature=0.7,
            max_tokens=500
        )

        # Step 3: Extract the assistant's reply.
        assistant_reply = response.choices[0].message.content

        # Step 4: Append the assistant's reply to the history for future context.
        self.messages.append({"role": "assistant", "content": assistant_reply})

        return assistant_reply

    def get_history(self):
        """Return the full conversation history."""
        return self.messages


# Test the agent with a multi-turn conversation
agent = SimpleAgent()

# Turn 1
print("USER: What is a Python decorator?")
print(f"AGENT: {agent.chat('What is a Python decorator?')}")
print()

# Turn 2 -- the agent should remember the previous question
print("USER: Can you show me a simple example?")
print(f"AGENT: {agent.chat('Can you show me a simple example?')}")
print()

# Show conversation history length
history = agent.get_history()
print(f"Conversation history: {len(history)} messages")
for msg in history:
    print(f"  [{msg['role']}] {msg['content'][:80]}{'...' if len(msg['content']) > 80 else ''}")

---
## Summary

In this session students learned the foundational skills for working with LLMs programmatically:

1. **API Connections** -- How to initialize the OpenAI client and make chat completion requests.
2. **Tokenization** -- How text is converted to tokens, why token counts matter for cost and context window limits, and how to use `tiktoken` to measure them.
3. **Model Parameters** -- How `temperature`, `top_p`, and `max_tokens` influence the style, creativity, and length of LLM outputs.
4. **System Messages & Personas** -- How the system message acts as a powerful control lever to change LLM behavior without changing the user prompt.
5. **LLM Pipelines** -- How to chain multiple LLM calls together so the output of one step feeds into the next, forming the basis for agentic workflows.

**Instructor Notes:**
- Encourage students to experiment with different prompts in the demos before moving to the tasks.
- For Task 2, discuss edge cases: what if the prompt has no periods? What about very long single sentences?
- For Task 4, discuss how the message history grows and what happens when it exceeds the context window.

**Up Next:** In Session 2, we will dive into advanced prompting techniques -- including few-shot prompting, chain-of-thought reasoning, and structured output generation -- that make LLMs more reliable and useful as the reasoning core of agentic systems.